# 01 - Training: PyTorch Baseline (Stanford Cars / Cars196)

This notebook establishes the **baseline model** for the vehicle vision project:

```
Stanford Cars -> preprocess -> EfficientNet-B0 (ImageNet pretrained) -> fine-tune on 196 classes -> checkpoint
```

The checkpoint produced here (`models/baseline/best.pt`) is the input for:

- `02_evaluation.ipynb` - full test-set evaluation (the baseline all later phases are compared against)
- `03_video_inference.ipynb` - video pipeline: detector -> crop -> this classifier -> annotated video

Run the notebooks in order. This notebook is self-contained and stores everything the later
notebooks need (class mapping, preprocessing, config) inside the checkpoint.

## 1. Environment

Report versions and hardware so results are reproducible, then fix the random seed.

In [ ]:
import os, sys, time, random, platform, textwrap

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset

%matplotlib inline

print("Python        :", sys.version.split()[0])
print("PyTorch       :", torch.__version__)
print("torchvision   :", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device        :", DEVICE, "(seed =", SEED, ")")

## 2. Configuration

All knobs live here. Paths are **not** hard-coded: `DATASET_ROOT` defaults to a relative
location and can be overridden with the `STANFORD_CARS_ROOT` environment variable, so the
same notebook runs on any machine without edits.

In [ ]:
from pathlib import Path

DATASET_ROOT    = Path(os.environ.get("STANFORD_CARS_ROOT", "../data/stanford_cars"))
MODELS_DIR      = Path(os.environ.get("MODELS_DIR", "../models"))
CHECKPOINT_PATH = MODELS_DIR / "baseline" / "best.pt"

IMAGE_SIZE    = 224
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2            # increase on a desktop CPU, keep 0 on Windows if dataloaders hang
NUM_CLASSES   = 196
VAL_FRACTION  = 0.1          # split off part of train as validation for checkpoint selection

print("DATASET_ROOT    :", DATASET_ROOT)
print("CHECKPOINT_PATH :", CHECKPOINT_PATH)
print("IMAGE_SIZE      :", IMAGE_SIZE)
print("BATCH_SIZE      :", BATCH_SIZE)
print("NUM_EPOCHS      :", NUM_EPOCHS)
print("LEARNING_RATE   :", LEARNING_RATE)
print("NUM_CLASSES     :", NUM_CLASSES)

## 3. Dataset

**Stanford Cars / Cars196**: 16,185 images, 196 classes (8,144 train / 8,041 test).

The dataset is **not** in this repository - download it separately and point
`DATASET_ROOT` at it. The original Stanford download URL is no longer reliable, so we do
**not** use `torchvision.datasets.StanfordCars(download=True)`. Instead this notebook
supports the two layouts that current mirrors provide:

**Layout A - official devkit files** (needs `scipy`):

```
stanford_cars/
+-- cars_meta.mat            class names
+-- cars_annos.mat           annotations for all 16,185 images (train + labeled test)
+-- cars_train/              8,144 images
+-- cars_test/               8,041 images
```

**Layout B - one folder per class** (e.g. the Kaggle mirror `jutrera/stanford-car-dataset-by-classes-folder`):

```
stanford_cars/
+-- train/<class name>/*.jpg
+-- test/<class name>/*.jpg
```

Both layouts end up as the same `(image path, label)` pairs, so the rest of the notebook
does not care which one you have.

In [ ]:
def discover_stanford_cars(root):
    """Find a Stanford Cars layout under `root`.

    Returns {"classes": [str], "train": [(path, label)], "test": [(path, label)]}
    """
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            "DATASET_ROOT does not exist: " + str(root)
            + "\nSet the STANFORD_CARS_ROOT environment variable or edit the configuration cell.")
    if (root / "cars_meta.mat").exists() and (root / "cars_annos.mat").exists():
        return _discover_devkit(root)
    if (root / "train").is_dir() and (root / "test").is_dir():
        return _discover_class_folders(root)
    raise FileNotFoundError(
        "No supported Stanford Cars layout under " + str(root) + "\n"
        + "Layout A (devkit):  cars_meta.mat, cars_annos.mat, cars_train/, cars_test/\n"
        + "Layout B (folders): train/<class name>/*.jpg, test/<class name>/*.jpg\n"
        + "Download the dataset manually (e.g. the Kaggle mirror "
        + "jutrera/stanford-car-dataset-by-classes-folder).")


def _discover_devkit(root):
    from scipy.io import loadmat  # only needed for this layout
    class_names = [str(c[0]) for c in loadmat(root / "cars_meta.mat")["class_names"].ravel()]
    train_samples, test_samples = [], []
    for a in loadmat(root / "cars_annos.mat")["annotations"].ravel():
        rel     = str(np.asarray(a["relative_im_path"]).ravel()[0])
        label   = int(np.asarray(a["class"]).ravel()[0]) - 1
        is_test = bool(int(np.asarray(a["test"]).ravel()[0]))
        (test_samples if is_test else train_samples).append((root / rel, label))
    return {"classes": class_names, "train": train_samples, "test": test_samples}


def _discover_class_folders(root):
    classes = sorted(p.name for p in (root / "train").iterdir() if p.is_dir())
    class_to_idx = {name: i for i, name in enumerate(classes)}
    result = {"classes": classes, "train": [], "test": []}
    for split in ("train", "test"):
        for class_dir in sorted((root / split).iterdir()):
            if not class_dir.is_dir():
                continue
            label = class_to_idx[class_dir.name]
            for pattern in ("*.jpg", "*.jpeg", "*.png"):
                for img_path in sorted(class_dir.glob(pattern)):
                    result[split].append((img_path, label))
    return result


class StanfordCarsDataset(Dataset):
    """Dataset over (path, label) pairs; transform is passed in so train and
    eval copies can use different preprocessing over the same images."""

    def __init__(self, samples, classes, transform=None):
        self.samples = samples
        self.classes = classes
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [ ]:
data = discover_stanford_cars(DATASET_ROOT)
classes       = data["classes"]
train_samples = data["train"]
test_samples  = data["test"]

assert len(classes) == NUM_CLASSES, "expected %d classes, found %d" % (NUM_CLASSES, len(classes))

print("train samples:", len(train_samples))
print("test samples :", len(test_samples))
print("num classes  :", len(classes))
print()
print("first 10 class names:")
for name in classes[:10]:
    print("  -", name)
print("  ...")

counts = np.bincount([label for _, label in train_samples], minlength=len(classes))
print()
print("images per class (train): min %d | median %d | max %d" % (counts.min(), int(np.median(counts)), counts.max()))

In [ ]:
# Look at a few raw training images (no augmentation applied here).
shown = random.Random(SEED).sample(train_samples, 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (path, label) in zip(axes.ravel(), shown):
    ax.imshow(Image.open(path))
    ax.set_title("\n".join(textwrap.wrap(classes[label], 26)), fontsize=9)
    ax.axis("off")
plt.suptitle("Random training images", y=1.02)
plt.tight_layout()
plt.show()

## 4. Preprocessing

Train and eval preprocessing differ (augmentation vs. deterministic), but both are built
from **one dictionary** (`PREPROCESSING`). That dictionary is stored in the checkpoint so
`02_evaluation.ipynb` and `03_video_inference.ipynb` can rebuild *exactly* the same
preprocessing - this is what keeps training and inference consistent later, including on
the edge device.

In [ ]:
from torchvision import transforms

MEAN = [0.485, 0.456, 0.406]   # ImageNet statistics
STD  = [0.229, 0.224, 0.225]

PREPROCESSING = {
    "image_size": IMAGE_SIZE,
    "mean": MEAN,
    "std": STD,
    "interpolation": "bilinear",
}

def build_transforms(preprocessing, augment):
    """Build the transform for one mode from a PREPROCESSING dict.

    augment=True  -> training: random crop + horizontal flip
    augment=False -> eval / inference: deterministic resize
    """
    size = preprocessing["image_size"]
    if augment:
        spatial = [transforms.RandomResizedCrop(size, scale=(0.6, 1.0)),
                   transforms.RandomHorizontalFlip()]
    else:
        spatial = [transforms.Resize((size, size))]
    return transforms.Compose(
        spatial
        + [transforms.ToTensor(),
           transforms.Normalize(preprocessing["mean"], preprocessing["std"])])

train_tf = build_transforms(PREPROCESSING, augment=True)
eval_tf  = build_transforms(PREPROCESSING, augment=False)
print("train transform:", train_tf)
print()
print("eval transform :", eval_tf)

## 5. Model

**EfficientNet-B0** pretrained on ImageNet: small (~5M parameters), accurate enough, and a
natural first candidate for later edge deployment (the distilled student + TensorRT will
be compared against this baseline).

The ImageNet classification head is replaced with a 196-way head. Everything else is kept
as-is and fine-tuned end to end.

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

MODEL_ARCH = "efficientnet_b0"

def build_model(num_classes=NUM_CLASSES, pretrained=True):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

model = build_model().to(DEVICE)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("architecture:", MODEL_ARCH)
print("parameters  : %.2f M (trainable %.2f M)" % (total / 1e6, trainable / 1e6))

## 6. Training

Stanford Cars only defines *train* and *test* splits. To keep the official test set clean
for `02_evaluation.ipynb` (it becomes the number all later phases are compared against), we
split **`VAL_FRACTION` of the training images off as validation** with a fixed seed and use
validation Top-1 accuracy for best-checkpoint selection.

Training is a plain supervised loop: cross-entropy loss, AdamW, cosine LR schedule. No
fancy tricks at this stage - this is the baseline.

In [ ]:
from torch.utils.data import Subset

val_len  = int(len(train_samples) * VAL_FRACTION)
perm     = list(range(len(train_samples)))
random.Random(SEED).shuffle(perm)
val_indices, train_indices = perm[:val_len], perm[val_len:]

# Two dataset instances over the same images but with different transforms.
train_ds = StanfordCarsDataset(train_samples, classes, transform=train_tf)
val_ds   = StanfordCarsDataset(train_samples, classes, transform=eval_tf)
test_ds  = StanfordCarsDataset(test_samples,  classes, transform=eval_tf)

pin = DEVICE.type == "cuda"
train_loader = DataLoader(Subset(train_ds, train_indices), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader   = DataLoader(Subset(val_ds, val_indices), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print("train/val split: %d / %d (VAL_FRACTION=%s, seed=%d)" % (len(train_indices), len(val_indices), VAL_FRACTION, SEED))


def accuracy(outputs, targets, topk=(1,)):
    """Per-sample Top-k accuracy in percent for each k in topk."""
    maxk = max(topk)
    _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)
    correct = pred.t().eq(targets[None])          # [maxk, batch]
    return [correct[:k].any(dim=0).float().mean().item() * 100.0 for k in topk]


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * targets.size(0)
        n += targets.size(0)
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Return (mean loss, Top-1 %, Top-5 %) over the loader."""
    model.eval()
    total_loss, top1, top5, n = 0.0, 0.0, 0.0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        outputs = model(images)
        total_loss += criterion(outputs, targets).item() * targets.size(0)
        a1, a5 = accuracy(outputs, targets, topk=(1, 5))
        top1 += a1 * targets.size(0)
        top5 += a5 * targets.size(0)
        n += targets.size(0)
    return total_loss / n, top1 / n, top5 / n

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_top1": [], "val_top5": [], "lr": []}
best = {"val_top1": -1.0, "val_top5": -1.0, "epoch": -1, "state_dict": None}

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_top1, val_top5 = evaluate(model, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top5"].append(val_top5)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    marker = ""
    if val_top1 > best["val_top1"]:
        best = {
            "val_top1": val_top1,
            "val_top5": val_top5,
            "epoch": epoch,
            # keep a CPU copy so the checkpoint is device-independent
            "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        }
        marker = "  <- best"

    scheduler.step()
    print("epoch %2d/%d | train loss %.4f | val loss %.4f | top1 %5.2f%% | top5 %5.2f%% | %5.1fs%s"
          % (epoch, NUM_EPOCHS, train_loss, val_loss, val_top1, val_top5, time.time() - t0, marker))

## 7. Training curves

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, NUM_EPOCHS + 1)

ax_loss.plot(epochs, history["train_loss"], marker="o", label="train loss")
ax_loss.plot(epochs, history["val_loss"], marker="o", label="val loss")
ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("cross-entropy loss")
ax_loss.set_title("Loss"); ax_loss.legend(); ax_loss.grid(alpha=0.3)

ax_acc.plot(epochs, history["val_top1"], marker="o", label="val top-1")
ax_acc.plot(epochs, history["val_top5"], marker="o", label="val top-5")
ax_acc.axvline(best["epoch"], color="gray", linestyle="--", label="best epoch")
ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("accuracy (%)")
ax_acc.set_title("Validation accuracy"); ax_acc.legend(); ax_acc.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Save checkpoint

The checkpoint contains everything needed to reproduce inference **without this notebook**:

| key | purpose |
|---|---|
| `model_arch` / `state_dict` | rebuild the architecture and load the weights |
| `classes` | index -> class-name mapping (label ids are list positions) |
| `preprocessing` | image size + normalization, reused by 02 and 03 |
| `training_config` | hyperparameters and seed |
| `metrics` / `history` | validation results of the best epoch |

`models/` holds weights only - **do not commit `.pt` files** (add `models/`, `data/` and
`outputs/` to `.gitignore`).

In [ ]:
checkpoint = {
    "format_version": 1,
    "dataset": "stanford_cars",
    "model_arch": MODEL_ARCH,
    "state_dict": best["state_dict"],
    "classes": classes,
    "num_classes": len(classes),
    "preprocessing": PREPROCESSING,
    "training_config": {
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "val_fraction": VAL_FRACTION,
        "seed": SEED,
        "optimizer": "adamw",
        "scheduler": "cosine",
    },
    "metrics": {
        "val_top1": best["val_top1"],
        "val_top5": best["val_top5"],
        "val_loss": history["val_loss"][best["epoch"] - 1],
        "epoch": best["epoch"],
    },
    "history": history,
    "torch_version": str(torch.__version__),
    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, CHECKPOINT_PATH)
print("saved      :", CHECKPOINT_PATH, "(%.1f MB)" % (CHECKPOINT_PATH.stat().st_size / 1e6))
print("best epoch :", best["epoch"])
print("val top-1  : %.2f%%" % best["val_top1"])
print("val top-5  : %.2f%%" % best["val_top5"])

## 9. Sanity-check inference

Reload the checkpoint **from disk** (exactly the way `02` and `03` will) and run it on a
few test images. This is only a quick check that the checkpoint is loadable and produces
sensible predictions - the real evaluation lives in `02_evaluation.ipynb` and the real
application is video inference in `03_video_inference.ipynb`.

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)  # our own checkpoint

model = build_model(num_classes=ckpt["num_classes"], pretrained=False).to(DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# Rebuild eval preprocessing from the checkpoint itself.
eval_tf = build_transforms(ckpt["preprocessing"], augment=False)
test_ds = StanfordCarsDataset(test_samples, ckpt["classes"], transform=eval_tf)


def short_name(name):
    return "\n".join(textwrap.wrap(name, 24))


picks = random.Random(SEED).sample(range(len(test_ds)), 8)
inv_mean = torch.tensor(ckpt["preprocessing"]["mean"])[:, None, None]
inv_std  = torch.tensor(ckpt["preprocessing"]["std"])[:, None, None]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.ravel(), picks):
    image, target = test_ds[i]
    with torch.no_grad():
        probs = torch.softmax(model(image[None].to(DEVICE)), dim=1)[0].cpu()
    conf, pred = probs.max(0)
    ok = pred.item() == target
    show = (image * inv_std + inv_mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(show)
    title = ("OK  " if ok else "WRONG\n") + "gt:   " + short_name(ckpt["classes"][target])
    title += "\npred: " + short_name(ckpt["classes"][pred.item()]) + " (%.1f%%)" % (conf.item() * 100)
    ax.set_title(title, fontsize=8, color="green" if ok else "red")
    ax.axis("off")
plt.suptitle("Sanity check: ground truth vs. prediction", y=1.02)
plt.tight_layout()
plt.show()

## Summary

- Baseline **EfficientNet-B0** fine-tuned on Stanford Cars (196 classes), best epoch
  selected by validation Top-1.
- Checkpoint with full metadata saved to `models/baseline/best.pt`.
- Next: `02_evaluation.ipynb` - test-set metrics, confusion matrix, model size/latency
  (the numbers the distilled and TensorRT models must beat).